In [2]:
!pip install pgmpy pandas numpy
import pandas as pd
import numpy as np

# Đọc dữ liệu từ Kaggle
df = pd.read_csv('heart.csv')

# Tiền xử lý: Phân nhóm các biến liên tục thành các danh mục rời rạc
df['age'] = pd.cut(df['age'], bins=[0, 45, 60, 100], labels=['Young', 'Middle', 'Senior'])
df['chol'] = pd.cut(df['chol'], bins=[0, 200, 240, 600], labels=['Normal', 'Borderline', 'High'])

# Giả sử file có cột 'target' (1: Bệnh, 0: Không) và các cột khác
print(df[['age', 'sex', 'chol', 'target']].head())
from pgmpy.models import DiscreteBayesianNetwork

# Định nghĩa các cạnh của đồ thị (Mũi tên đi từ Nguyên nhân -> Kết quả)
model = DiscreteBayesianNetwork([
    ('age', 'chol'),            # Tuổi tác ảnh hưởng đến Cholesterol
    ('sex', 'chol'),            # Giới tính ảnh hưởng đến Cholesterol
    ('age', 'target'),    # Tuổi tác ảnh hưởng trực tiếp đến Bệnh tim (target)
    ('chol', 'target')    # Cholesterol ảnh hưởng đến Bệnh tim (target)
])
from pgmpy.estimators import MaximumLikelihoodEstimator

# Khớp (fit) dữ liệu vào mô hình để tính toán các CPTs
model.fit(df)

# In thử Bảng xác suất có điều kiện của node Bệnh tim (target)
print("Bảng xác suất của Bệnh tim (target) dựa trên Age và Chol:")
print(model.get_cpds('target'))
from pgmpy.inference import VariableElimination

infer = VariableElimination(model)

# Kịch bản 1: Dự đoán xác suất bị bệnh tim của một người đàn ông lớn tuổi (Senior)
q1 = infer.query(variables=['target'],
                 evidence={'age': 'Senior', 'sex': 1})
print("Xác suất mắc bệnh tim (Senior, Nam):")
print(q1)

# Kịch bản 2: Suy luận ngược (Diagnostic Reasoning)
# Nếu biết một người đã CÓ bệnh tim (target = 1), xác suất người đó có mức Cholesterol cao là bao nhiêu?
q2 = infer.query(variables=['chol'],
                 evidence={'target': 1})
print("\nXác suất mức Cholesterol của bệnh nhân bị bệnh tim:")
print(q2)

      age  sex        chol  target
0  Senior    1  Borderline       1
1   Young    1        High       1
2   Young    0  Borderline       1
3  Middle    1  Borderline       1
4  Middle    0        High       1
Bảng xác suất của Bệnh tim (target) dựa trên Age và Chol:
+-----------+---------------------+-----+--------------------+
| age       | age(Middle)         | ... | age(Young)         |
+-----------+---------------------+-----+--------------------+
| chol      | chol(Borderline)    | ... | chol(Normal)       |
+-----------+---------------------+-----+--------------------+
| target(0) | 0.45454545454545453 | ... | 0.3333333333333333 |
+-----------+---------------------+-----+--------------------+
| target(1) | 0.5454545454545454  | ... | 0.6666666666666666 |
+-----------+---------------------+-----+--------------------+
Xác suất mắc bệnh tim (Senior, Nam):
+-----------+---------------+
| target    |   phi(target) |
+===========+===============+
| target(0) |        0.5492 |
+-------